# Stage 3C.01 — setup and preflight
Reset-only audit; no policy checkpoint is loaded.

In [ ]:
import json,os,subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; PY=Path.home()/"venv-stage1-id/bin/python"; P=Path.home()/"LIBERO-plus"; S1=Path.home()/"stage1"; S3B=Path.home()/"stage3b"; OUT=Path.home()/"stage3c"; OUT.mkdir(exist_ok=True)
required=(R,P,P/"libero/libero/assets",PY,S1/"stage1_resolved_variants.csv",S3B/"stage3b_object_layout_manifest.csv",S3B/"stage3b_episode_results.csv",S3B/"stage3b_reused_stage3_controls_audit.csv")
missing=[str(p) for p in required if not p.exists()]
if missing: raise SystemExit(f"STOP: missing prerequisites: {missing}")
env=os.environ.copy(); env.update({"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg"})
subprocess.run([str(PY),"-m","pytest","-q",str(R/"async_vla_benchmark/tests")],cwd=R,env=env,check=True)
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.validate_stage3b","--manifest",str(S3B/"stage3b_object_layout_manifest.csv"),"--output-dir",str(S3B),"--reuse-audit",str(S3B/"stage3b_reused_stage3_controls_audit.csv")],cwd=R,env=env,check=True)
gpu="1"; line=subprocess.run(["nvidia-smi",f"--id={gpu}","--query-gpu=name,memory.total,memory.used,utilization.gpu,driver_version","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True).stdout.strip(); print(line)
name,total,used,util,driver=[x.strip() for x in line.split(',')]; assert "A100" in name
if int(used)>=500 or int(util)>=5: raise SystemExit("STOP: select an idle physical A100")
provenance={"repository_sha":subprocess.run(["git","-C",str(R),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(),"libero_plus_sha":subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(),"gpu":line,"policy_inference_executed":False}
(OUT/"stage3c_preflight_environment.json").write_text(json.dumps(provenance,indent=2)+"\n")
print("PASS: Stage 3C preflight complete; completed Stage 3B validated")